# || NEMO Benchmarking .tiff generator ||
© Konstantinos Andreadis 2024 (PhD in the Roux Lab & Salbreux Lab at UNIGE, Switzerland)

In [ ]:
%load_ext autoreload
%autoreload 2

# Import custom module scripts
from module_scripts import analysis, datahandler, visuals
import numpy as np
import os
import trimesh
from json import dump as jdump

In [ ]:
PARAMS = {
    'num_filaments': 900,
    'vesicle_intensity': 0.05,
    'filament_intensity': 1.0,
    'steps': 40,
    'dt': 0.015,
    'grid_size': 512,
    'radius': 200,
    'blur_sigma': 1.0,
    'vec_length': 20
}


def get_director(p, defects):
    z_denom = np.where(1 + p[:, 2] == 0, 1e-9, 1 + p[:, 2])
    w_pts = (p[:, 0] + 1j * p[:, 1]) / z_denom
    w_defects = (defects[:, 0] + 1j * defects[:, 1]) / (1 + defects[:, 2])
    psi = sum(0.5 * np.angle(w_pts - wd) for wd in w_defects)
    denom = 1 + p[:, 0] ** 2 / z_denom ** 2 + p[:, 1] ** 2 / z_denom ** 2
    u, v = p[:, 0] / z_denom, p[:, 1] / z_denom
    dp_du = np.stack([2 * (1 - u ** 2 + v ** 2), -4 * u * v, -4 * u], axis=-1) / (denom ** 2)[:, None]
    dp_dv = np.stack([-4 * u * v, 2 * (1 + u ** 2 - v ** 2), -4 * v], axis=-1) / (denom ** 2)[:, None]
    dp_du /= np.linalg.norm(dp_du, axis=-1, keepdims=True)
    dp_dv /= np.linalg.norm(dp_dv, axis=-1, keepdims=True)
    n_3d = np.cos(psi)[:, None] * dp_du + np.sin(psi)[:, None] * dp_dv
    return n_3d / np.linalg.norm(n_3d, axis=-1, keepdims=True)


def generate_vesicle(cfg):
    v = np.array([[1, 1, 1], [1, -1, -1], [-1, 1, -1], [-1, -1, 1]], dtype=float)
    defects = v / np.linalg.norm(v, axis=1, keepdims=True)

    seeds = trimesh.creation.icosphere(subdivisions=3).vertices
    np.random.shuffle(seeds)
    lines = []

    for p0 in seeds[:cfg['num_filaments']]:
        line = [p0]
        for direction in [1, -1]:
            p, last_n = p0.copy(), None
            for _ in range(cfg['steps']):
                n = get_director(p[None, :], defects)[0]
                if last_n is not None and np.dot(n, last_n) < 0: n = -n
                last_n, p = n, p + direction * cfg['dt'] * n
                p /= np.linalg.norm(p)
                line.append(p.copy()) if direction == 1 else line.insert(0, p.copy())
        lines.append(np.array(line))

    vol = np.zeros((cfg['grid_size'],) * 3, dtype=np.float32)
    c, r = cfg['grid_size'] // 2, cfg['radius']
    z, y, x = np.ogrid[:cfg['grid_size'], :cfg['grid_size'], :cfg['grid_size']]
    membrane = np.abs(np.sqrt((x - c) ** 2 + (y - c) ** 2 + (z - c) ** 2) - r) <= 1.5
    vol[membrane] = cfg['vesicle_intensity']
    for line in lines:
        vox = (line * r + c).astype(int)
        vox = vox[(vox >= 0).all(axis=1) & (vox < cfg['grid_size']).all(axis=1)]
        vol[vox[:, 2], vox[:, 1], vox[:, 0]] = cfg['filament_intensity']
    defect_pos = (defects * r + c)[:, ::-1]
    vol = analysis.gaussian_blur(img=vol, sigma=cfg['blur_sigma'], renorm=True)
    vol *= 255
    vol = np.clip(vol, 0, 255).astype(np.uint8)
    return vol, defect_pos


z_stack, defect_pts = generate_vesicle(PARAMS)

In [ ]:
visuals.plot_img(img=z_stack, scale=(1, 1, 1), unit="px", max_proj=False)
visuals.plot_img(img=z_stack, scale=(1, 1, 1), unit="px", max_proj=True, cmap="Greens")
datahandler.save_tiff(array=z_stack,
                      filepath='/Users/andreadi/Physbio Dropbox/Konstantinos Andreadis/Academic/Data/!nemo_figure_runs/simulated_vesicle.tif',
                      img_unit="um", img_scale=(1, 1, 1), scalar_type=np.uint8)


def save_params(path, params, name):
    file_path = os.path.join(path, f"{name}.json")
    with open(file_path, 'w') as f:
        jdump(params, f)
    print(f">> Saved parameters to {file_path} !")


save_params(path='/Users/andreadi/Physbio Dropbox/Konstantinos Andreadis/Academic/Data/!nemo_figure_runs/',
            params=PARAMS, name="simulated_vesicle")
datahandler.save_array(array=defect_pts, name="simulated_vesicle", header="x,y,z",
                       folderpath="/Users/andreadi/Physbio Dropbox/Konstantinos Andreadis/Academic/Data/!nemo_figure_runs/")